# YaRN vs native RoPE on needle-in-a-haystack

Interactive driver for the harness in this repo, for a free Kaggle (2xT4) or
Colab (T4) session. Every run appends to a JSONL, so a killed session resumes
instead of restarting.

Run the cells in order. The kernel equivalents (`kaggle/*/`) run the same code
headlessly with the grid baked in; use those if you want unattended runs.

In [ ]:
# 1) Get the code. Works whether this repo sits in /kaggle/working, /content or
#    anywhere else, as long as run.py is reachable.
import os, sys, pathlib

CANDIDATES = [os.getcwd(), "/kaggle/working/yarn-haystack", "/content/yarn-haystack"]
REPO = os.environ.get("NIAH_REPO") or next(
    (p for p in CANDIDATES if pathlib.Path(p, "run.py").exists()), None
)
if REPO is None:
    raise SystemExit(
        "put run.py/niah.py/analyze.py somewhere reachable, or set NIAH_REPO.\n"
        "  git clone https://github.com/Nikita-prog-art/yarn-haystack.git"
    )
os.chdir(REPO)
sys.path.insert(0, REPO)
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
print("repo:", REPO, "| work:", WORK)

In [ ]:
# 2) Install only what is missing. Kaggle ships a CUDA-matched torch, and
#    upgrading matplotlib/datasets provokes conflicts with preinstalled tools.
import importlib.metadata as md, subprocess, sys

def version(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

def too_old(v, floor):
    return v is None or tuple(int(x) for x in v.split(".")[:2]) < floor

to_install = []
if too_old(version("transformers"), (4, 51)):
    to_install.append("transformers>=4.51")
for p in ("accelerate", "datasets", "bitsandbytes"):
    if version(p) is None:
        to_install.append(p)
if to_install:
    print("pip install:", " ".join(to_install), flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *to_install], check=True)

for p in ("transformers", "torch", "accelerate", "datasets", "bitsandbytes", "matplotlib"):
    print(f"  {p:14s} {version(p)}")

import torch
print(f"\ntorch {torch.__version__} | cuda {torch.cuda.is_available()} | gpus {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    pr = torch.cuda.get_device_properties(i)
    print(f"  gpu{i}: {pr.name}, {pr.total_memory / 2**30:.1f} GiB")
assert torch.cuda.is_available(), "no CUDA: this kernel was pushed without a GPU"
_v = version("transformers")
assert _v and tuple(int(x) for x in _v.split(".")[:2]) >= (4, 51), f"need transformers>=4.51, have {_v}"

In [ ]:
# 4) THE EXPERIMENT CONFIG. Change these lines, nothing else.
#
#    Wave 1 (one T4, cheapest path to a number):
#        MODEL = "Qwen/Qwen2.5-7B-Instruct"   # native 32k, 56 KiB KV/token
#    Wave 2 (needs 2xT4 + vLLM TP=2: Qwen3 KV is 144 KiB/token => 18 GiB at 128k):
#        MODEL = "Qwen/Qwen3-8B"              # native 32k (config says 40960), 144 KiB/token
#
#    Do NOT use Qwen3.5-*, Qwen3-*-Instruct-2507, Qwen3-Next-* or Qwen3-Coder-*:
#    they are natively 256k, so at <=128k there is no extrapolation failure for
#    YaRN to fix and the delta would be zero by construction. run.py blocks them.
MODEL    = "Qwen/Qwen2.5-7B-Instruct"
ORIG_MAX = 32768            # the TRAINED prompt window, NOT config.json's 40960
FACTOR   = 4.0              # -> 131072; Qwen advise matching the target (2.0 for 64k)
LENGTHS  = "4096,8192,16384,32768,65536,131072"
DEPTHS   = "0.0,0.1,0.25,0.5,0.75,0.9,1.0"
EXTRA    = ["--load-in-4bit", "--attn", "sdpa"]   # identical for BOTH arms, always
TAG      = MODEL.split("/")[-1].lower().replace(".", "") + f"_f{FACTOR:g}"
print(TAG, LENGTHS)

In [ ]:
# 3) Preflight: internet (the usual cause of a one-second death) and the corpus.
import os, urllib.request
import niah

try:
    print("huggingface.co ->", urllib.request.urlopen("https://huggingface.co", timeout=20).status)
except Exception as exc:
    raise SystemExit(f"no internet in this kernel: {exc!r} (Kaggle needs phone verification for that)")

text = niah.load_corpus(None, os.path.join(os.getcwd(), "cache"))
print(f"corpus OK: {len(text)/1e6:.1f}M chars cached")

In [ ]:
import subprocess, sys, os, collections
RESULTS = f"{WORK}/results"; os.makedirs(RESULTS, exist_ok=True)

def niah(*args):
    """Run the harness as a subprocess, streaming its output live.

    -u is not decoration: without it the child's stdout is block-buffered through
    a pipe, so a 40-minute 128k case looks frozen. On failure the last 40 lines go
    INTO the exception, because a bare CalledProcessError hides the actual error.
    """
    cmd = [sys.executable, "-u", "run.py", *args]
    print("$", " ".join(cmd), flush=True)
    tail = collections.deque(maxlen=40)
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end="", flush=True)
            tail.append(line.rstrip())
        p.wait()
    if p.returncode:
        raise RuntimeError(f"run.py exited with {p.returncode}. Last lines of its output:\n  "
                           + "\n  ".join(tail))

In [ ]:
# 5) SMOKE TEST FIRST: tiny model, one length.
#    run.py raises if rope_scaling did not stick, but read the logged config too.
niah("--model", "Qwen/Qwen2.5-1.5B-Instruct", "--arm", "yarn", "--factor", "4",
     "--original-max", "32768", "--lengths", "4096", "--depths", "0.5",
     "--max-new-tokens", "24", "--out", f"{RESULTS}/smoke.jsonl")

In [ ]:
# 6) ARM 1 - native RoPE, extrapolated past its window. Expect collapse at 64k/128k.
#    Not a throwaway arm: yarn@128k - base@128k IS the YaRN effect.
niah("--model", MODEL, "--arm", "base",
     "--lengths", LENGTHS, "--depths", DEPTHS, "--max-new-tokens", "64",
     "--out", f"{RESULTS}/base_{TAG}.jsonl", *EXTRA)

In [ ]:
# 7) ARM 2 - same weights, YaRN. Byte-identical setup otherwise: same EXTRA, same grid.
niah("--model", MODEL, "--arm", "yarn", "--factor", str(FACTOR),
     "--original-max", str(ORIG_MAX),
     "--lengths", LENGTHS, "--depths", DEPTHS, "--max-new-tokens", "64",
     "--out", f"{RESULTS}/yarn_{TAG}.jsonl", *EXTRA)

In [ ]:
# 8) Report: summary.md + heatmaps.png + CSVs, incl. the delta-vs-base table.
import glob, subprocess, sys, os
from IPython.display import Image, Markdown, display

res = sorted(glob.glob(f"{RESULTS}/*.jsonl"))
print("result files:", res)
if not res:
    raise SystemExit("no results yet - run the arms above")
subprocess.run([sys.executable, "analyze.py", *res, "--outdir", f"{WORK}/report_{TAG}"], check=True)
display(Markdown(open(f"{WORK}/report_{TAG}/summary.md", encoding="utf-8").read()))
if os.path.exists(f"{WORK}/report_{TAG}/heatmaps.png"):
    display(Image(f"{WORK}/report_{TAG}/heatmaps.png"))

### Reading the result

- `base` at 64k/128k collapsing to ~0 while `yarn` holds up = YaRN doing its job.
- Check the **4k/8k/16k row**. Qwen's own card warns that static YaRN "may
  potentially degrade model performance" on short texts: a drop there is real, and
  it is the price of the 128k gain.
- Depth matters. NIAH tends to break in the middle before it breaks at the edges,
  so a flat-looking heatmap row at every length is worth more than one number.

### If 7B @ 128k OOMs

`--load-in-4bit --attn sdpa` needs ~11 GiB (NF4 weights + 7 GiB KV). If it still OOMs:
- `Qwen/Qwen2.5-3B-Instruct` in fp16 (~10.3 GiB at 128k): same protocol, weaker model;
- `--device-map auto` to shard across Kaggle's 2xT4 (slow, but it fits);
- `--backend vllm --tensor-parallel-size 2` (fastest, iff vLLM installs cleanly).
Never mix these flags across arms - a 4-bit `base` vs fp16 `yarn` measures quantization.

### Persisting across sessions

`/kaggle/working` survives a session stop. The JSONL is append-only and fsync'd,
so run.py skips finished cases automatically: just re-run the same cell. Sync with
`--upload <user>/yarn-niah` for belt and braces.